# CanAI Café — Forecasting Model
**Owner:** Member C  
**Input:** `../data/clean/data_clean.csv`  
**Output:** `../outputs/forecast_results.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

## 1. Load & Prepare Time Series

In [ ]:
df = pd.read_csv('../data/clean/data_clean.csv', parse_dates=['date'])  # update column name if needed

# Aggregate to monthly total revenue
monthly = df.groupby(df['date'].dt.to_period('M'))['revenue'].sum().reset_index()  # update column names
monthly['date'] = monthly['date'].dt.to_timestamp()
monthly.head()

## 2. Visualise the Series

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(monthly['date'], monthly['revenue'])
plt.title('Monthly Revenue')
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.tight_layout()
plt.show()

## 3. Train / Test Split

In [ ]:
split = int(len(monthly) * 0.8)
train = monthly.iloc[:split]
test  = monthly.iloc[split:]
print(f'Train: {len(train)} months | Test: {len(test)} months')

## 4. Model — Prophet
*(Swap this section for SARIMA or another model if preferred)*

In [ ]:
from prophet import Prophet

# Prophet expects columns named 'ds' and 'y'
train_prophet = train.rename(columns={'date': 'ds', 'revenue': 'y'})

model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
model.fit(train_prophet)

## 5. Evaluate on Holdout Set

In [ ]:
future_test = model.make_future_dataframe(periods=len(test), freq='MS')
forecast_test = model.predict(future_test)

preds = forecast_test.tail(len(test))['yhat'].values
actuals = test['revenue'].values

mae  = mean_absolute_error(actuals, preds)
rmse = root_mean_squared_error(actuals, preds)
mape = (abs((actuals - preds) / actuals)).mean() * 100

print(f'MAE:  {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'MAPE: {mape:.2f}%')

## 6. Generate 6-Month Forecast

In [ ]:
# Refit on full dataset, then forecast 6 months ahead
full_prophet = monthly.rename(columns={'date': 'ds', 'revenue': 'y'})
model_full = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
model_full.fit(full_prophet)

future = model_full.make_future_dataframe(periods=6, freq='MS')
forecast = model_full.predict(future)

forecast_6m = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(6)
forecast_6m.columns = ['month', 'forecasted_revenue', 'lower_bound', 'upper_bound']
forecast_6m

## 7. Save Forecast Output

In [ ]:
forecast_6m.to_csv('../outputs/forecast_results.csv', index=False)
print('Saved: ../outputs/forecast_results.csv')